# 📚 04. 심화 — pandas 고급 활용을 위한 Python

이 노트북은 **pandas를 능숙하게 다루기 위한** 고급 Python 개념을 다룹니다.

## 학습 목표
| 개념 | pandas 연결 포인트 |
|------|-------------------|
| 이터레이터 | `iterrows()`, `itertuples()` + 성능 비교 |
| *args / **kwargs | pandas 함수 시그니처 이해, 유연한 함수 설계 |
| 데코레이터 | `df.pipe()` 메서드 체이닝, 전처리 파이프라인 |

> **전제**: 01_필수 ~ 03_중요 노트북을 모두 완료해주세요.

In [ ]:
# =============================================
# 공통 데이터 준비 (50개 샘플 - 성능 비교용)
# =============================================

import pandas as pd
import numpy as np
import time
import random

# 설명: 성능 비교를 위해 더 큰 데이터셋 생성
# 50개 행으로 구성 (이터레이터 속도 차이가 나타나도록)

random.seed(42)  # 재현성을 위해 시드 고정
np.random.seed(42)

상품_목록 = ['노트북', '마우스', '키보드', '모니터', '헤드셋', '웹캠', '스피커', '허브', '패드', '케이블']
카테고리_목록 = ['IT기기', 'IT기기', 'IT기기', 'IT기기', '음향기기', '영상기기', '음향기기', 'IT기기', 'IT기기', 'IT기기']
지역_목록 = ['서울', '부산', '대구', '인천', '광주', '대전']

n = 50  # 행 수
인덱스 = list(range(n))
상품명들 = [상품_목록[i % len(상품_목록)] for i in range(n)]
카테고리들 = [카테고리_목록[i % len(카테고리_목록)] for i in range(n)]

df = pd.DataFrame({
    '상품명':   상품명들,
    '카테고리': 카테고리들,
    '가격':     np.random.randint(10000, 2000000, n),
    '수량':     np.random.randint(1, 100, n),
    '지역':     [random.choice(지역_목록) for _ in range(n)],
    '평점':     np.round(np.random.uniform(3.0, 5.0, n), 1),
})
df['매출'] = df['가격'] * df['수량']

print(f'데이터 준비 완료! ({len(df)}행 {len(df.columns)}열)')
print(df.head())

---
## 1장. 이터레이터 (Iterator)

이터레이터는 **한 번에 하나씩 값을 꺼내주는** 객체입니다.  
리스트는 모든 데이터를 메모리에 올리지만, 이터레이터는 **필요할 때만** 데이터를 생성합니다.

> **pandas 연결**:  
> `df.iterrows()` — 행을 (인덱스, Series) 튜플로 순회  
> `df.itertuples()` — 행을 Named Tuple로 순회 (더 빠름)  
> 벡터화 연산 — 이터레이터보다 훨씬 빠름 (권장)

In [ ]:
# =============================================
# 1-1. 이터레이터 기본 개념
# =============================================

# 설명: 이터레이터는 __iter__(), __next__() 메서드를 가진 객체
#       for 루프는 내부적으로 이터레이터를 사용

# 일반 리스트 → 이터레이터 변환
숫자_리스트 = [10, 20, 30, 40, 50]
이터레이터 = iter(숫자_리스트)  # iter()로 이터레이터 생성

# next()로 하나씩 꺼내기
print('=== next()로 하나씩 꺼내기 ===')
print('1번째:', next(이터레이터))  # → 10
print('2번째:', next(이터레이터))  # → 20
print('3번째:', next(이터레이터))  # → 30

# for 루프는 내부적으로 이렇게 동작!
print('\n=== for 루프 = 내부적으로 iter() + next() ===')
for 값 in [10, 20, 30]:
    print(f'  값: {값}')

# 제너레이터: 이터레이터를 쉽게 만드는 방법 (yield 사용)
def 숫자_생성기(최대값):
    """0부터 최대값까지 하나씩 생성하는 제너레이터"""
    현재 = 0
    while 현재 < 최대값:
        yield 현재  # yield: 값을 반환하고 일시 정지
        현재 += 1

print('\n=== 제너레이터 사용 ===')
for 숫자 in 숫자_생성기(5):
    print(f'  생성: {숫자}', end=' ')
print()

In [ ]:
# =============================================
# 1-2. df.iterrows() — 행을 딕셔너리처럼 순회
# =============================================

# 설명: iterrows()는 (인덱스, Series) 튜플을 반환
# 주의: 각 행이 Series로 변환되므로 느림 (타입 변환 오버헤드)
# 용도: 행마다 복잡한 조건 분기가 필요한 경우

print('=== iterrows() 사용 예시 (처음 5행) ===')
for 인덱스, 행 in df.head(5).iterrows():
    # 행은 딕셔너리처럼 컬럼명으로 접근
    상품명 = 행['상품명']
    가격 = 행['가격']
    매출 = 행['매출']
    print(f'  [{인덱스:2d}] {상품명:<6} | 가격 {가격:>10,}원 | 매출 {매출:>12,}원')

# iterrows()로 조건부 처리
print('\n=== iterrows()로 고매출 상품 태그 추가 ===')
태그_목록 = []
for _, 행 in df.iterrows():
    if 행['매출'] >= df['매출'].quantile(0.75):  # 상위 25%
        태그_목록.append('★ 고매출')
    else:
        태그_목록.append('')

df['태그'] = 태그_목록
print(df[df['태그'] == '★ 고매출'][['상품명', '매출', '태그']].head())

In [ ]:
# =============================================
# 1-3. df.itertuples() — 더 빠른 행 순회
# =============================================

# 설명: itertuples()는 Named Tuple을 반환 → iterrows()보다 빠름
# 사용법: 행.컬럼명 으로 점(.) 표기법 접근 (행['컬럼명'] 아님!)
# 주의: 컬럼명에 공백이나 특수문자가 있으면 접근 불가 (getattr 사용)

print('=== itertuples() 사용 예시 (처음 5행) ===')
for 행 in df.head(5).itertuples():
    # Named Tuple: 행.Index, 행.상품명, 행.가격 등으로 접근
    print(f'  [{행.Index:2d}] {행.상품명:<6} | {행.가격:>10,}원 | 평점 {행.평점}')

print('\n=== itertuples() 반환 타입 확인 ===')
첫행 = next(df.itertuples())
print(f'타입: {type(첫행)}')
print(f'첫 행: {첫행}')

In [ ]:
# =============================================
# 1-4. 성능 비교: for vs iterrows vs itertuples vs 벡터화
# =============================================

# 설명: 같은 작업을 4가지 방법으로 수행하여 속도 비교
# 작업: 매출이 평균 이상인 상품 수 카운트
# pandas 연결: 벡터화 연산이 압도적으로 빠름!

# 1만 개 데이터로 테스트
df_큰 = pd.DataFrame({
    '가격': np.random.randint(10000, 2000000, 10000),
    '수량': np.random.randint(1, 100, 10000),
})
df_큰['매출'] = df_큰['가격'] * df_큰['수량']
평균_매출 = df_큰['매출'].mean()

# 방법 1: 일반 for 루프 (리스트 변환 후)
시작 = time.time()
카운트1 = sum(1 for m in df_큰['매출'].tolist() if m > 평균_매출)
시간1 = time.time() - 시작

# 방법 2: iterrows()
시작 = time.time()
카운트2 = sum(1 for _, 행 in df_큰.iterrows() if 행['매출'] > 평균_매출)
시간2 = time.time() - 시작

# 방법 3: itertuples()
시작 = time.time()
카운트3 = sum(1 for 행 in df_큰.itertuples() if 행.매출 > 평균_매출)
시간3 = time.time() - 시작

# 방법 4: pandas 벡터화 (권장!)
시작 = time.time()
카운트4 = (df_큰['매출'] > 평균_매출).sum()
시간4 = time.time() - 시작

print('=== 성능 비교 (10,000행, 평균 이상 매출 카운트) ===')
print(f'  1. for 루프:    {시간1:.4f}초   (결과: {카운트1})')
print(f'  2. iterrows():  {시간2:.4f}초   (결과: {카운트2})')
print(f'  3. itertuples():{시간3:.4f}초   (결과: {카운트3})')
print(f'  4. 벡터화:      {시간4:.6f}초 (결과: {카운트4}) ← 권장!')
print(f'\n  → 벡터화가 iterrows()보다 약 {시간2/max(시간4,0.0001):.0f}배 빠름')
print('\n📌 결론: 가능하면 항상 벡터화 연산 사용!')
print('   iterrows/itertuples는 꼭 필요할 때만 (복잡한 분기, 외부 API 호출 등)')

In [ ]:
# =============================================
# 1-5. 실용적인 이터레이터 활용: 배치 처리
# =============================================

# 설명: 대용량 데이터를 작은 단위로 나눠 처리하는 배치 처리
# pandas 연결: 메모리 제한이 있을 때 청크 단위로 CSV 읽기에 활용

def 배치_이터레이터(데이터프레임, 배치크기):
    """DataFrame을 배치크기로 나누는 제너레이터"""
    총행수 = len(데이터프레임)
    for 시작 in range(0, 총행수, 배치크기):
        끝 = min(시작 + 배치크기, 총행수)
        yield 데이터프레임.iloc[시작:끝]  # 배치 단위로 yield

# 배치 단위 처리
print('=== 배치 처리 (배치크기=10) ===')
배치_결과 = []
for 배치번호, 배치_df in enumerate(배치_이터레이터(df, 10), 1):
    배치_매출합 = 배치_df['매출'].sum()
    배치_결과.append({'배치': 배치번호, '행수': len(배치_df), '매출합': 배치_매출합})
    print(f'  배치 {배치번호}: {len(배치_df)}행 처리 → 매출합 {배치_매출합:,}원')

print(f'\n전체 배치 수: {len(배치_결과)}개')
print(f'전체 매출합: {sum(b["매출합"] for b in 배치_결과):,}원')
print(f'검증 (pandas): {df["매출"].sum():,}원')

# pandas chunking: 대용량 CSV를 청크로 읽기
print('\n=== pandas CSV 청크 읽기 (실제 활용법) ===')
print('코드 예시:')
print('  # 100만 행 CSV를 10만 행씩 읽어 처리')
print('  for 청크 in pd.read_csv("대용량.csv", chunksize=100000):')
print('      처리(청크)')

---
## 2장. *args와 **kwargs

함수가 **가변 개수의 인수**를 받을 수 있게 하는 문법입니다.

```python
*args   → 위치 인수를 튜플로 받음  (positional arguments)
**kwargs → 키워드 인수를 딕셔너리로 받음 (keyword arguments)
```

> **pandas 연결**:  
> `df.groupby()`, `df.merge()`, `df.pivot_table()` 등 pandas 함수들이  
> `**kwargs`를 사용하여 유연한 매개변수를 지원합니다.

In [ ]:
# =============================================
# 2-1. *args: 가변 위치 인수
# =============================================

# 설명: *args는 임의 개수의 위치 인수를 튜플로 받음
# 사용법: 함수 호출 시 몇 개를 전달해도 OK!

def 합계(*숫자들):
    """임의 개수의 숫자를 받아 합계 반환"""
    # 설명: 숫자들은 튜플 타입
    print(f'  받은 값들: {숫자들} (타입: {type(숫자들).__name__})')
    return sum(숫자들)

print('=== *args 사용 예시 ===')
print('합계(1, 2):', 합계(1, 2))
print('합계(1, 2, 3, 4, 5):', 합계(1, 2, 3, 4, 5))
print('합계(10, 20, 30, 40, 50, 60):', 합계(10, 20, 30, 40, 50, 60))

# 실용 예제: 여러 DataFrame 합치기
def 여러_DataFrame_합치기(*데이터프레임들):
    """여러 DataFrame을 세로로 합치는 함수"""
    print(f'  합칠 DataFrame 개수: {len(데이터프레임들)}')
    return pd.concat(데이터프레임들, ignore_index=True)

df_a = df.iloc[0:10]   # 처음 10행
df_b = df.iloc[10:20]  # 다음 10행
df_c = df.iloc[20:30]  # 다음 10행

print('\n=== 여러 DataFrame 합치기 ===')
합친_df = 여러_DataFrame_합치기(df_a, df_b, df_c)
print(f'합친 결과: {len(합친_df)}행')

In [ ]:
# =============================================
# 2-2. **kwargs: 가변 키워드 인수
# =============================================

# 설명: **kwargs는 임의 개수의 키워드 인수를 딕셔너리로 받음
# pandas 연결: pandas 함수 내부에서 이 방식으로 옵션을 전달받음

def 설정_출력(**설정들):
    """임의 키워드 인수를 딕셔너리로 받아 출력"""
    # 설명: 설정들은 딕셔너리 타입
    print(f'  받은 설정들: {설정들}')
    for 키, 값 in 설정들.items():
        print(f'    {키} = {값}')

print('=== **kwargs 사용 예시 ===')
설정_출력(색상='파랑', 크기=14, 굵게=True)
설정_출력(제목='판매 분석', 작성자='홍길동')

# 실용 예제: pandas 함수에 설정 전달
def CSV_읽기(파일경로, **pandas_옵션):
    """CSV를 읽을 때 추가 옵션을 **kwargs로 전달"""
    # **pandas_옵션을 pd.read_csv에 그대로 전달!
    return pd.read_csv(파일경로, **pandas_옵션)

# 다양한 옵션 전달
print('\n=== pandas 옵션을 **kwargs로 전달 ===')
df_읽기 = CSV_읽기('판매_pandas.csv',
                    encoding='utf-8-sig',
                    usecols=['상품명', '가격', '매출'])
print(df_읽기.head(3))

In [ ]:
# =============================================
# 2-3. *args와 **kwargs 함께 사용
# =============================================

# 설명: 위치 인수와 키워드 인수를 동시에 가변으로 받기
# 순서 규칙: def 함수(일반매개변수, *args, **kwargs)
# pandas 연결: pandas의 많은 함수가 이 패턴을 사용

def 유연한_집계(데이터프레임, *컬럼들, **집계_옵션):
    """
    여러 컬럼의 집계를 유연하게 수행
    *컬럼들: 집계할 컬럼명들
    **집계_옵션: groupby 기준 등 옵션
    """
    # 결과를 담을 딕셔너리
    결과 = {}
    
    for 컬럼 in 컬럼들:
        if 컬럼 not in 데이터프레임.columns:
            print(f'  주의: "{컬럼}" 컬럼 없음')
            continue
        
        결과[컬럼] = {
            '합계': 데이터프레임[컬럼].sum(),
            '평균': 데이터프레임[컬럼].mean(),
            '최대': 데이터프레임[컬럼].max(),
        }
        
        # 그룹별 집계 옵션 처리
        if 'groupby' in 집계_옵션:
            그룹컬럼 = 집계_옵션['groupby']
            결과[f'{컬럼}_그룹별'] = 데이터프레임.groupby(그룹컬럼)[컬럼].sum()
    
    return 결과


# 사용 예시
print('=== 유연한 집계 함수 사용 ===')
결과 = 유연한_집계(df, '가격', '매출', '수량', groupby='카테고리')

for 키, 값 in 결과.items():
    if isinstance(값, dict):
        print(f'\n[{키}]')
        for 통계명, 통계값 in 값.items():
            print(f'  {통계명}: {통계값:,.0f}')
    else:
        print(f'\n[{키}]')
        print(값)

In [ ]:
# =============================================
# 2-4. 언패킹 연산자 (* / **)
# =============================================

# 설명: *는 리스트를, **는 딕셔너리를 펼쳐서 전달
# pandas 연결: 미리 정의된 옵션 딕셔너리를 pandas 함수에 전달할 때 유용

# 리스트 언패킹 (*)
컬럼_목록 = ['상품명', '가격', '매출']
print('=== 리스트 언패킹 ===')
print(df[컬럼_목록].head(3))   # 직접 리스트 전달
print(df[*컬럼_목록,].head(3)) # * 언패킹 (Python 3.9+)

# 딕셔너리 언패킹 (**)
읽기_옵션 = {
    'encoding': 'utf-8-sig',
    'usecols': ['상품명', '가격'],
}

print('\n=== 딕셔너리 언패킹으로 pandas 옵션 전달 ===')
df_옵션 = pd.read_csv('판매_pandas.csv', **읽기_옵션)  # ** 로 딕셔너리 전달
print(df_옵션.head(3))

# 집계 함수 옵션도 딕셔너리로 관리
그룹집계_옵션 = {'by': '카테고리', 'sort': True}
결과 = df.groupby(**그룹집계_옵션)['매출'].sum()
print('\n카테고리별 매출:')
print(결과)

---
## 3장. 데코레이터 (Decorator)

데코레이터는 **기존 함수를 수정하지 않고** 추가 기능을 입히는 고급 문법입니다.  
`@데코레이터명` 문법으로 함수 위에 붙여서 사용합니다.

```python
@타이머        # 이것이 데코레이터
def 분석():
    ...
```

> **pandas 연결**:  
> `df.pipe(함수)` — 데코레이터처럼 함수를 연결하는 메서드  
> 전처리 파이프라인을 `pipe()`로 연결하면 데코레이터 패턴과 유사

In [ ]:
# =============================================
# 3-1. 데코레이터 기본 구조
# =============================================

# 설명: 데코레이터는 함수를 인수로 받아 새 함수를 반환하는 함수
# 구조: 외부함수(감싸는 함수) → 내부함수(실행 추가) → 원본함수 호출

import functools

def 타이머(함수):
    """함수 실행 시간을 측정하는 데코레이터"""
    @functools.wraps(함수)  # 원본 함수의 이름/문서 유지
    def 래퍼(*args, **kwargs):
        시작 = time.time()
        결과 = 함수(*args, **kwargs)  # 원본 함수 실행
        종료 = time.time()
        print(f'  [{함수.__name__}] 실행시간: {(종료-시작)*1000:.2f}ms')
        return 결과
    return 래퍼

# 데코레이터 적용
@타이머
def 데이터_정렬(데이터프레임, 기준='매출'):
    """DataFrame을 기준 컬럼으로 정렬"""
    return 데이터프레임.sort_values(기준, ascending=False)

@타이머  
def 상위_추출(데이터프레임, n=10):
    """상위 n개 행 추출"""
    return 데이터프레임.head(n)

# 실행하면 자동으로 시간 측정!
print('=== 타이머 데코레이터 적용 ===')
정렬된_df = 데이터_정렬(df)
상위_df = 상위_추출(df, n=5)

print(상위_df[['상품명', '매출']].head())

In [ ]:
# =============================================
# 3-2. 실용 데코레이터: 로깅 + 검증
# =============================================

# 설명: 여러 유용한 데코레이터 구현
# pandas 연결: 데이터 분석 함수에 자동으로 검증 추가

def DataFrame_검증기(함수):
    """함수 입력이 DataFrame인지 검증하는 데코레이터"""
    @functools.wraps(함수)
    def 래퍼(*args, **kwargs):
        # 첫 번째 인수가 DataFrame인지 확인
        if args and isinstance(args[0], pd.DataFrame):
            입력_df = args[0]
            print(f'  입력: DataFrame ({len(입력_df)}행 {len(입력_df.columns)}열)')
        else:
            raise TypeError('첫 번째 인수는 DataFrame이어야 합니다!')
        
        결과 = 함수(*args, **kwargs)
        
        if isinstance(결과, pd.DataFrame):
            print(f'  출력: DataFrame ({len(결과)}행 {len(결과.columns)}열)')
        
        return 결과
    return 래퍼


def 로거(함수이름):
    """함수 실행을 로그로 남기는 팩토리 데코레이터"""
    def 데코레이터(함수):
        @functools.wraps(함수)
        def 래퍼(*args, **kwargs):
            print(f'🔵 [{함수이름}] 시작')
            결과 = 함수(*args, **kwargs)
            print(f'🟢 [{함수이름}] 완료')
            return 결과
        return 래퍼
    return 데코레이터


@로거('결측값 처리')
@DataFrame_검증기
def 결측값_처리(데이터프레임):
    """결측값을 평균으로 대체하는 전처리 함수"""
    df_복사 = 데이터프레임.copy()
    for col in df_복사.select_dtypes(include='number').columns:
        df_복사[col] = df_복사[col].fillna(df_복사[col].mean())
    return df_복사


print('=== 데코레이터 체인 실행 ===')
처리된_df = 결측값_처리(df)

In [ ]:
# =============================================
# 3-3. pandas pipe() — 데코레이터 패턴
# =============================================

# 설명: pipe()는 함수를 체이닝하는 pandas 메서드
#       데코레이터처럼 DataFrame에 처리 단계를 순서대로 적용
# 문법: df.pipe(함수1).pipe(함수2, 인수).pipe(함수3)

# 파이프라인 단계별 함수 정의
def 이상값_제거(데이터프레임, 컬럼, 범위배수=3):
    """평균에서 범위배수 * 표준편차 이상인 값 제거"""
    평균 = 데이터프레임[컬럼].mean()
    표준편차 = 데이터프레임[컬럼].std()
    최소 = 평균 - 범위배수 * 표준편차
    최대 = 평균 + 범위배수 * 표준편차
    필터 = (데이터프레임[컬럼] >= 최소) & (데이터프레임[컬럼] <= 최대)
    결과 = 데이터프레임[필터]
    print(f'  이상값 제거: {len(데이터프레임)} → {len(결과)}행')
    return 결과

def 매출_계산(데이터프레임):
    """매출 컬럼 재계산"""
    df_새 = 데이터프레임.copy()
    df_새['매출'] = df_새['가격'] * df_새['수량']
    print(f'  매출 재계산 완료')
    return df_새

def 등급_추가(데이터프레임):
    """매출 기준 등급 컬럼 추가"""
    df_새 = 데이터프레임.copy()
    q75 = df_새['매출'].quantile(0.75)
    q25 = df_새['매출'].quantile(0.25)
    df_새['등급'] = df_새['매출'].apply(
        lambda x: 'A' if x >= q75 else ('B' if x >= q25 else 'C')
    )
    print(f'  등급 추가 완료')
    return df_새

# pipe()로 체이닝: 마치 데코레이터처럼 단계별 적용
print('=== df.pipe() 체이닝 파이프라인 ===')
결과_df = (
    df
    .pipe(이상값_제거, '가격', 범위배수=2)  # 1단계: 이상값 제거
    .pipe(매출_계산)                         # 2단계: 매출 재계산
    .pipe(등급_추가)                         # 3단계: 등급 추가
)

print('\n최종 결과:')
print(결과_df[['상품명', '가격', '매출', '등급']].head(10))

In [ ]:
# =============================================
# 3-4. 데코레이터 vs pipe() 비교
# =============================================

# 설명: 두 가지 방법의 차이를 명확히 이해

print('=== 데코레이터 패턴 ===')
print('  @데코레이터A')
print('  @데코레이터B')
print('  def 함수(): ...')
print('  → 함수 정의 시점에 기능 결합')

print('\n=== pipe() 패턴 ===')
print('  df.pipe(함수A).pipe(함수B).pipe(함수C)')
print('  → 실행 시점에 순서대로 적용')

print('\n=== 각 방법의 장단점 ===')
비교표 = [
    ('특징', '데코레이터', 'pipe()'),
    ('적용 대상', '모든 함수', 'DataFrame만'),
    ('코드 위치', '함수 정의부', '실행부'),
    ('재사용성', '함수 수준', '체이닝 수준'),
    ('가독성', '함수가 복잡해짐', '흐름이 명확'),
    ('언제 사용?', '로깅, 검증, 캐싱', '데이터 전처리 흐름'),
]
for 행 in 비교표:
    print(f'  {행[0]:<12} | {행[1]:<20} | {행[2]}')

---
## 4장. 종합 실습 — 전체 4단계 개념 통합

In [ ]:
# =============================================
# 4-1. 종합 실습: 분석 파이프라인 클래스
# =============================================

# 설명: 4개 노트북의 모든 개념을 통합한 최종 실습
# - 클래스 (03_중요)
# - 이터레이터 / 배치 처리 (04_심화)
# - *args / **kwargs (04_심화)
# - 데코레이터 (04_심화)
# - 파일 I/O + 예외 처리 (03_중요)

def 실행_시간_기록(함수):
    """실행 시간을 측정하는 데코레이터"""
    @functools.wraps(함수)
    def 래퍼(self, *args, **kwargs):
        시작 = time.time()
        결과 = 함수(self, *args, **kwargs)
        소요 = time.time() - 시작
        print(f'  ⏱  {함수.__name__}: {소요*1000:.1f}ms')
        return 결과
    return 래퍼


class 통합_분석_파이프라인:
    """4개 노트북 개념을 모두 통합한 분석 파이프라인"""
    
    def __init__(self, 데이터프레임):
        # 클래스 기초 (03_중요)
        self.원본_df = 데이터프레임.copy()
        self.현재_df = 데이터프레임.copy()
        self.단계_기록 = []  # 처리 단계 기록
        print(f'파이프라인 초기화: {len(self.현재_df)}행')
    
    @실행_시간_기록  # 데코레이터 적용 (04_심화)
    def 전처리(self, *제거할_컬럼들, **fillna_옵션):
        """
        *args: 제거할 컬럼명들
        **kwargs: fillna 옵션 (컬럼명=채울값)
        """
        # *args 활용: 여러 컬럼 한 번에 제거
        if 제거할_컬럼들:
            존재하는_컬럼 = [c for c in 제거할_컬럼들 if c in self.현재_df.columns]
            self.현재_df = self.현재_df.drop(columns=존재하는_컬럼)
            print(f'    컬럼 제거: {존재하는_컬럼}')
        
        # **kwargs 활용: 컬럼별 결측값 대체
        for 컬럼, 대체값 in fillna_옵션.items():
            if 컬럼 in self.현재_df.columns:
                self.현재_df[컬럼] = self.현재_df[컬럼].fillna(대체값)
                print(f'    {컬럼} 결측값 → {대체값}로 대체')
        
        self.단계_기록.append('전처리')
        return self  # 메서드 체이닝 지원
    
    @실행_시간_기록
    def 특성_추가(self):
        """파생 특성(Feature) 컬럼 추가"""
        # lambda 활용 (02_핵심)
        if '매출' not in self.현재_df.columns and '가격' in self.현재_df.columns:
            self.현재_df['매출'] = self.현재_df['가격'] * self.현재_df['수량']
        
        if '매출' in self.현재_df.columns:
            q75 = self.현재_df['매출'].quantile(0.75)
            self.현재_df['고매출_여부'] = self.현재_df['매출'].apply(
                lambda x: True if x >= q75 else False
            )
        
        self.단계_기록.append('특성추가')
        return self
    
    @실행_시간_기록
    def 배치_집계(self, 배치크기=10):
        """이터레이터로 배치 단위 집계 (04_심화)"""
        # 제너레이터로 배치 처리
        def 배치_생성(데이터프레임, 크기):
            for i in range(0, len(데이터프레임), 크기):
                yield 데이터프레임.iloc[i:i+크기]
        
        배치_통계 = []
        for 배치 in 배치_생성(self.현재_df, 배치크기):
            if '매출' in 배치.columns:
                배치_통계.append({'매출합': 배치['매출'].sum(), '행수': len(배치)})
        
        self.배치_결과 = pd.DataFrame(배치_통계)
        self.단계_기록.append('배치집계')
        return self
    
    @실행_시간_기록
    def 결과_저장(self, 파일경로, **저장_옵션):
        """결과를 CSV로 저장 (03_중요: 파일 I/O + 예외 처리)"""
        try:
            기본_옵션 = {'index': False, 'encoding': 'utf-8-sig'}
            기본_옵션.update(저장_옵션)  # 사용자 옵션으로 업데이트
            self.현재_df.to_csv(파일경로, **기본_옵션)
            print(f'    저장 완료: {파일경로}')
        except Exception as e:
            print(f'    저장 실패: {e}')
        
        self.단계_기록.append('결과저장')
        return self
    
    def 요약_출력(self):
        """전체 처리 요약"""
        print('\n==============================')
        print('     파이프라인 실행 요약')
        print('==============================')
        print(f'처리 단계: {" → ".join(self.단계_기록)}')
        print(f'최종 데이터: {len(self.현재_df)}행 {len(self.현재_df.columns)}열')
        
        if '매출' in self.현재_df.columns:
            print(f'총 매출: {self.현재_df["매출"].sum():,}원')
            print(f'평균 매출: {self.현재_df["매출"].mean():,.0f}원')
        
        if hasattr(self, '배치_결과'):
            print(f'배치 처리: {len(self.배치_결과)}개 배치')


# 파이프라인 실행
print('=== 통합 파이프라인 실행 ===')
파이프라인 = 통합_분석_파이프라인(df)
(
    파이프라인
    .전처리('태그', '가격등급', 평점=4.0)  # *args로 컬럼 제거, **kwargs로 fillna
    .특성_추가()
    .배치_집계(배치크기=10)
    .결과_저장('최종_분석.csv')
)
파이프라인.요약_출력()

In [ ]:
# =============================================
# 4-2. 최종 분석 결과 확인
# =============================================

print('=== 최종 분석 결과 (처음 10행) ===')
출력_컬럼 = ['상품명', '카테고리', '가격', '수량', '매출', '고매출_여부']
출력_컬럼 = [c for c in 출력_컬럼 if c in 파이프라인.현재_df.columns]
print(파이프라인.현재_df[출력_컬럼].head(10))

print('\n=== 카테고리별 고매출 비율 ===')
if '고매출_여부' in 파이프라인.현재_df.columns:
    고매출_비율 = 파이프라인.현재_df.groupby('카테고리')['고매출_여부'].mean() * 100
    for 카테고리, 비율 in 고매출_비율.items():
        print(f'  {카테고리}: {비율:.1f}%')

print('\n=== 배치 처리 결과 ===')
print(파이프라인.배치_결과)

---
## 정리 — 전체 4개 노트북 요약

| 노트북 | 핵심 개념 | pandas 연결 |
|--------|-----------|-------------|
| **01_필수** | 리스트, 딕셔너리, for문, 조건문 | DataFrame 생성, 필터링 원리 |
| **02_핵심** | def/lambda, 슬라이싱, 인덱싱, 컴프리헨션 | apply/map, iloc/loc, 열 가공 |
| **03_중요** | 클래스, 파일 I/O, 예외 처리 | 객체 구조 이해, read_csv, 안전한 코드 |
| **04_심화** | 이터레이터, *args/**kwargs, 데코레이터 | iterrows/itertuples, API 이해, pipe |

### pandas 핵심 규칙 (복습)

1. **벡터화 우선**: `for` 대신 `df['열'].apply()` 또는 벡터 연산
2. **iloc vs loc**: 숫자 위치 → `iloc`, 이름/레이블 → `loc`
3. **결측값 방어**: `fillna()`, `dropna()`, `pd.to_numeric(errors='coerce')`
4. **파이프라인**: `df.pipe(함수1).pipe(함수2)` 로 깔끔한 데이터 흐름
5. **iterrows 주의**: 느리므로 꼭 필요할 때만, 대부분은 `apply()`로 대체 가능

### 다음 단계 — pandas 본격 학습!
이제 Python 전제 지식이 완성되었습니다. pandas를 배울 준비가 되었습니다! 🎉
- `groupby()` / `agg()` — 그룹별 집계
- `merge()` / `join()` — 데이터 결합
- `pivot_table()` — 피벗 테이블
- `resample()` — 시계열 데이터 처리
- `matplotlib` / `seaborn` — 시각화